# MMVC ONNX Export Workflow

このノートブックは、訓練済みのMMVCモデルをONNX形式にエクスポートし、最適化を行うためのワークフローです。ONNXモデルは異なるプラットフォームでの推論に使用できます。

## 1. 必要なライブラリのインポート

In [ ]:
import os
import json
import torch
import onnx
import onnxruntime as ort
import numpy as np
import time
from onnxsim import simplify
import matplotlib.pyplot as plt

# MMVC modules
import sys
sys.path.append('..')

from models import SynthesizerTrn
from text import text_to_sequence
from text.symbols import symbols
import utils
import commons

print("ライブラリのインポートが完了しました。")
print(f"PyTorch バージョン: {torch.__version__}")
print(f"ONNX バージョン: {onnx.__version__}")
print(f"ONNX Runtime バージョン: {ort.__version__}")

## 2. 設定とモデルの読み込み

In [ ]:
# 設定ファイルの読み込み
config_path = "../configs/baseconfig.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

print("設定を読み込みました:")
print(f"- モデル名: {config['model_name']}")
print(f"- サンプリング周波数: {config['sampling_rate']} Hz")

# デバイス設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

# モデルパス
model_path = "../logs/G_latest.pth"  # 訓練済みモデルのパス

# モデルの初期化
net_g = SynthesizerTrn(
    len(symbols),
    config["filter_length"] // 2 + 1,
    config["segment_size"] // config["hop_length"],
    n_speakers=config.get("n_speakers", 0),
    **config["model"]
).to(device)

# チェックポイントの読み込み
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device)
    net_g.load_state_dict(checkpoint['model'])
    print(f"モデルを読み込みました: {model_path}")
    print(f"エポック: {checkpoint.get('epoch', 'N/A')}")
else:
    print(f"警告: モデルファイルが見つかりません: {model_path}")
    print("先に3_Train_MMVC.ipynbでモデルを訓練してください。")

# 推論モードに設定
net_g.eval()
print("モデルを推論モードに設定しました。")

## 3. ONNX エクスポート準備

In [ ]:
# エクスポート用のダミー入力を準備
def prepare_dummy_inputs():
    """
    ONNX エクスポート用のダミー入力を準備
    """
    # テキスト入力（短いサンプル）
    sample_text = "こんにちは"
    stn_tst = text_to_sequence(sample_text, ["japanese_cleaners"])
    
    # ダミー入力の作成
    x = torch.LongTensor(stn_tst).unsqueeze(0).to(device)
    x_lengths = torch.LongTensor([len(stn_tst)]).to(device)
    
    # スピーカーID（マルチスピーカーの場合）
    if config.get("n_speakers", 0) > 0:
        sid = torch.LongTensor([0]).to(device)
    else:
        sid = None
    
    # ノイズパラメータ
    noise_scale = torch.FloatTensor([0.667]).to(device)
    noise_scale_w = torch.FloatTensor([0.8]).to(device)
    length_scale = torch.FloatTensor([1.0]).to(device)
    
    return x, x_lengths, sid, noise_scale, noise_scale_w, length_scale

# ダミー入力の準備
dummy_inputs = prepare_dummy_inputs()
x, x_lengths, sid, noise_scale, noise_scale_w, length_scale = dummy_inputs

print("ダミー入力を準備しました:")
print(f"- テキスト長: {x.shape}")
print(f"- テキスト長さ: {x_lengths}")
if sid is not None:
    print(f"- スピーカーID: {sid}")
print(f"- ノイズスケール: {noise_scale}")

## 4. 推論専用モデルの作成

In [ ]:
class MMVCInferenceModel(torch.nn.Module):
    """
    ONNX エクスポート用の推論専用モデル
    """
    def __init__(self, model, n_speakers=0):
        super().__init__()
        self.model = model
        self.n_speakers = n_speakers
    
    def forward(self, x, x_lengths, sid=None, noise_scale=0.667, noise_scale_w=0.8, length_scale=1.0):
        """
        推論専用のforward関数
        """
        with torch.no_grad():
            if self.n_speakers > 0 and sid is not None:
                audio = self.model.infer(x, x_lengths, sid=sid, 
                                       noise_scale=noise_scale, 
                                       noise_scale_w=noise_scale_w, 
                                       length_scale=length_scale)[0]
            else:
                audio = self.model.infer(x, x_lengths, 
                                       noise_scale=noise_scale, 
                                       noise_scale_w=noise_scale_w, 
                                       length_scale=length_scale)[0]
            return audio

# 推論専用モデルの作成
inference_model = MMVCInferenceModel(net_g, config.get("n_speakers", 0))
inference_model.eval()

print("推論専用モデルを作成しました。")

# PyTorchモデルでのテスト実行
print("\nPyTorchモデルでテスト実行中...")
start_time = time.time()
with torch.no_grad():
    if sid is not None:
        torch_output = inference_model(x, x_lengths, sid, noise_scale, noise_scale_w, length_scale)
    else:
        torch_output = inference_model(x, x_lengths, None, noise_scale, noise_scale_w, length_scale)
pytorch_time = time.time() - start_time

print(f"PyTorch推論時間: {pytorch_time:.4f}秒")
print(f"出力音声長: {torch_output.shape}")

## 5. ONNX エクスポート実行

In [ ]:
# ONNXエクスポート設定
onnx_path = "../models/mmvc_model.onnx"
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

# 入力名の定義
if config.get("n_speakers", 0) > 0:
    input_names = ["x", "x_lengths", "sid", "noise_scale", "noise_scale_w", "length_scale"]
    dummy_input = (x, x_lengths, sid, noise_scale, noise_scale_w, length_scale)
else:
    input_names = ["x", "x_lengths", "noise_scale", "noise_scale_w", "length_scale"]
    dummy_input = (x, x_lengths, noise_scale, noise_scale_w, length_scale)

output_names = ["audio"]

# 動的次元の設定
if config.get("n_speakers", 0) > 0:
    dynamic_axes = {
        "x": {0: "batch_size", 1: "sequence_length"},
        "x_lengths": {0: "batch_size"},
        "sid": {0: "batch_size"},
        "audio": {0: "batch_size", 2: "audio_length"}
    }
else:
    dynamic_axes = {
        "x": {0: "batch_size", 1: "sequence_length"},
        "x_lengths": {0: "batch_size"},
        "audio": {0: "batch_size", 2: "audio_length"}
    }

print("ONNX エクスポート開始...")
try:
    torch.onnx.export(
        inference_model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        verbose=False
    )
    print(f"ONNX エクスポート完了: {onnx_path}")
except Exception as e:
    print(f"ONNX エクスポートエラー: {e}")
    print("エラーの詳細を確認し、必要に応じてモデルを修正してください。")

## 6. ONNX モデルの検証

In [ ]:
if os.path.exists(onnx_path):
    print("ONNX モデルの検証中...")
    
    # ONNX モデルの読み込みと検証
    try:
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        print("✓ ONNX モデルの構造は正常です。")
        
        # モデル情報の表示
        print(f"\nONNX モデル情報:")
        print(f"- IR バージョン: {onnx_model.ir_version}")
        print(f"- オペレータセット: {onnx_model.opset_import[0].version}")
        print(f"- グラフ名: {onnx_model.graph.name}")
        print(f"- 入力数: {len(onnx_model.graph.input)}")
        print(f"- 出力数: {len(onnx_model.graph.output)}")
        print(f"- ノード数: {len(onnx_model.graph.node)}")
        
        # 入力情報
        print(f"\n入力情報:")
        for input_info in onnx_model.graph.input:
            print(f"- {input_info.name}: {input_info.type}")
        
        # 出力情報
        print(f"\n出力情報:")
        for output_info in onnx_model.graph.output:
            print(f"- {output_info.name}: {output_info.type}")
            
    except Exception as e:
        print(f"ONNX モデル検証エラー: {e}")
else:
    print("ONNX モデルファイルが見つかりません。")

## 7. ONNX モデルの最適化

In [ ]:
if os.path.exists(onnx_path):
    print("ONNX モデルの最適化中...")
    
    try:
        # モデルの簡略化
        onnx_model = onnx.load(onnx_path)
        print(f"最適化前のノード数: {len(onnx_model.graph.node)}")
        
        # 簡略化実行
        model_simplified, check = simplify(onnx_model)
        
        if check:
            print(f"最適化後のノード数: {len(model_simplified.graph.node)}")
            
            # 最適化されたモデルの保存
            optimized_path = onnx_path.replace(".onnx", "_optimized.onnx")
            onnx.save(model_simplified, optimized_path)
            print(f"✓ 最適化済みモデルを保存: {optimized_path}")
            
            # ファイルサイズ比較
            original_size = os.path.getsize(onnx_path) / (1024*1024)
            optimized_size = os.path.getsize(optimized_path) / (1024*1024)
            reduction = (1 - optimized_size/original_size) * 100
            
            print(f"\nファイルサイズ比較:")
            print(f"- 元のモデル: {original_size:.2f} MB")
            print(f"- 最適化後: {optimized_size:.2f} MB")
            print(f"- 削減率: {reduction:.1f}%")
        else:
            print("モデルの簡略化に失敗しました。")
            
    except Exception as e:
        print(f"最適化エラー: {e}")
        print("最適化をスキップして元のモデルを使用します。")

## 8. ONNX Runtime での推論テスト

In [ ]:
def test_onnx_inference(model_path):
    """
    ONNX Runtime での推論テスト
    """
    try:
        # ONNX Runtime セッションの作成
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if torch.cuda.is_available() else ['CPUExecutionProvider']
        session = ort.InferenceSession(model_path, providers=providers)
        
        print(f"ONNX Runtime セッションを作成しました。")
        print(f"使用プロバイダー: {session.get_providers()}")
        
        # 入力データの準備
        input_dict = {
            "x": x.cpu().numpy(),
            "x_lengths": x_lengths.cpu().numpy(),
            "noise_scale": noise_scale.cpu().numpy(),
            "noise_scale_w": noise_scale_w.cpu().numpy(),
            "length_scale": length_scale.cpu().numpy()
        }
        
        if config.get("n_speakers", 0) > 0 and sid is not None:
            input_dict["sid"] = sid.cpu().numpy()
        
        # 推論実行
        print("\nONNX推論実行中...")
        start_time = time.time()
        onnx_outputs = session.run(["audio"], input_dict)
        onnx_time = time.time() - start_time
        
        onnx_audio = onnx_outputs[0]
        
        print(f"ONNX推論時間: {onnx_time:.4f}秒")
        print(f"出力音声長: {onnx_audio.shape}")
        
        # PyTorchとONNXの結果比較
        torch_audio_np = torch_output.cpu().numpy()
        
        # 形状の確認
        if torch_audio_np.shape == onnx_audio.shape:
            diff = np.abs(torch_audio_np - onnx_audio)
            max_diff = np.max(diff)
            mean_diff = np.mean(diff)
            
            print(f"\n精度比較:")
            print(f"- 最大誤差: {max_diff:.6f}")
            print(f"- 平均誤差: {mean_diff:.6f}")
            
            if max_diff < 1e-3:
                print("✓ PyTorchとONNXの出力は十分に一致しています。")
            else:
                print("⚠ PyTorchとONNXの出力に大きな差があります。")
        else:
            print(f"⚠ 出力形状が異なります: PyTorch {torch_audio_np.shape} vs ONNX {onnx_audio.shape}")
        
        # 速度比較
        speedup = pytorch_time / onnx_time
        print(f"\n速度比較:")
        print(f"- PyTorch: {pytorch_time:.4f}秒")
        print(f"- ONNX: {onnx_time:.4f}秒")
        print(f"- 速度向上: {speedup:.2f}x")
        
        return session, onnx_audio
        
    except Exception as e:
        print(f"ONNX推論テストエラー: {e}")
        return None, None

# ONNX推論テスト実行
if os.path.exists(onnx_path):
    onnx_session, onnx_result = test_onnx_inference(onnx_path)
    
    # 最適化モデルもテスト
    optimized_path = onnx_path.replace(".onnx", "_optimized.onnx")
    if os.path.exists(optimized_path):
        print("\n" + "="*50)
        print("最適化モデルのテスト:")
        onnx_session_opt, onnx_result_opt = test_onnx_inference(optimized_path)
else:
    print("ONNX モデルが見つかりません。先にエクスポートを実行してください。")

## 9. ベンチマークテスト

In [ ]:
def benchmark_models(num_runs=10):
    """
    PyTorchとONNXモデルのベンチマーク比較
    """
    if not os.path.exists(onnx_path):
        print("ONNX モデルが見つかりません。")
        return
    
    print(f"ベンチマークテスト開始 ({num_runs} 回実行)...")
    
    # PyTorch ベンチマーク
    pytorch_times = []
    for i in range(num_runs):
        start_time = time.time()
        with torch.no_grad():
            if sid is not None:
                _ = inference_model(x, x_lengths, sid, noise_scale, noise_scale_w, length_scale)
            else:
                _ = inference_model(x, x_lengths, None, noise_scale, noise_scale_w, length_scale)
        pytorch_times.append(time.time() - start_time)
    
    # ONNX ベンチマーク
    if onnx_session:
        onnx_times = []
        input_dict = {
            "x": x.cpu().numpy(),
            "x_lengths": x_lengths.cpu().numpy(),
            "noise_scale": noise_scale.cpu().numpy(),
            "noise_scale_w": noise_scale_w.cpu().numpy(),
            "length_scale": length_scale.cpu().numpy()
        }
        
        if config.get("n_speakers", 0) > 0 and sid is not None:
            input_dict["sid"] = sid.cpu().numpy()
        
        for i in range(num_runs):
            start_time = time.time()
            _ = onnx_session.run(["audio"], input_dict)
            onnx_times.append(time.time() - start_time)
    
    # 結果の分析
    pytorch_mean = np.mean(pytorch_times)
    pytorch_std = np.std(pytorch_times)
    
    print(f"\nベンチマーク結果:")
    print(f"PyTorch:")
    print(f"  平均: {pytorch_mean:.4f} ± {pytorch_std:.4f} 秒")
    print(f"  最小: {np.min(pytorch_times):.4f} 秒")
    print(f"  最大: {np.max(pytorch_times):.4f} 秒")
    
    if onnx_session and len(onnx_times) > 0:
        onnx_mean = np.mean(onnx_times)
        onnx_std = np.std(onnx_times)
        speedup = pytorch_mean / onnx_mean
        
        print(f"\nONNX Runtime:")
        print(f"  平均: {onnx_mean:.4f} ± {onnx_std:.4f} 秒")
        print(f"  最小: {np.min(onnx_times):.4f} 秒")
        print(f"  最大: {np.max(onnx_times):.4f} 秒")
        print(f"  速度向上: {speedup:.2f}x")
        
        # グラフ表示
        plt.figure(figsize=(12, 6))
        
        plt.subplot(1, 2, 1)
        plt.bar(['PyTorch', 'ONNX'], [pytorch_mean, onnx_mean], 
                yerr=[pytorch_std, onnx_std], capsize=5)
        plt.ylabel('推論時間 (秒)')
        plt.title('平均推論時間比較')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(range(num_runs), pytorch_times, 'o-', label='PyTorch', alpha=0.7)
        plt.plot(range(num_runs), onnx_times, 's-', label='ONNX', alpha=0.7)
        plt.xlabel('実行回数')
        plt.ylabel('推論時間 (秒)')
        plt.title('推論時間の推移')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

# ベンチマーク実行
benchmark_models()

## 10. ONNX モデルの詳細分析

In [ ]:
def analyze_onnx_model(model_path):
    """
    ONNX モデルの詳細分析
    """
    if not os.path.exists(model_path):
        print(f"モデルファイルが見つかりません: {model_path}")
        return
    
    print(f"\n=== ONNX モデル分析: {os.path.basename(model_path)} ===")
    
    # ファイルサイズ
    file_size = os.path.getsize(model_path) / (1024*1024)
    print(f"ファイルサイズ: {file_size:.2f} MB")
    
    # モデルの読み込み
    model = onnx.load(model_path)
    
    # 基本情報
    print(f"IR バージョン: {model.ir_version}")
    print(f"オペレータセット: {model.opset_import[0].version}")
    print(f"グラフ名: {model.graph.name}")
    
    # ノード分析
    node_types = {}
    for node in model.graph.node:
        if node.op_type in node_types:
            node_types[node.op_type] += 1
        else:
            node_types[node.op_type] = 1
    
    print(f"\n総ノード数: {len(model.graph.node)}")
    print("ノードタイプ別カウント:")
    for op_type, count in sorted(node_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {op_type}: {count}")
    
    # 入出力情報
    print(f"\n入力数: {len(model.graph.input)}")
    for i, input_info in enumerate(model.graph.input):
        print(f"  入力{i+1}: {input_info.name}")
    
    print(f"\n出力数: {len(model.graph.output)}")
    for i, output_info in enumerate(model.graph.output):
        print(f"  出力{i+1}: {output_info.name}")
    
    # パラメータ数推定（初期化子から）
    total_params = 0
    for initializer in model.graph.initializer:
        param_count = 1
        for dim in initializer.dims:
            param_count *= dim
        total_params += param_count
    
    print(f"\n推定パラメータ数: {total_params:,}")

# 元のモデルの分析
if os.path.exists(onnx_path):
    analyze_onnx_model(onnx_path)

# 最適化モデルの分析
optimized_path = onnx_path.replace(".onnx", "_optimized.onnx")
if os.path.exists(optimized_path):
    analyze_onnx_model(optimized_path)

## 11. デプロイメント用ファイル生成

In [ ]:
def create_deployment_package():
    """
    デプロイメント用のパッケージを作成
    """
    deploy_dir = "../deployment"
    os.makedirs(deploy_dir, exist_ok=True)
    
    # 設定ファイルのコピー
    import shutil
    shutil.copy(config_path, os.path.join(deploy_dir, "config.json"))
    
    # ONNXモデルのコピー
    if os.path.exists(onnx_path):
        shutil.copy(onnx_path, deploy_dir)
    
    optimized_path = onnx_path.replace(".onnx", "_optimized.onnx")
    if os.path.exists(optimized_path):
        shutil.copy(optimized_path, deploy_dir)
    
    # デプロイメント用のPythonスクリプト作成
    deployment_script = f'''
#!/usr/bin/env python3
"""
MMVC ONNX Inference Script
デプロイメント用の推論スクリプト
"""

import os
import json
import numpy as np
import onnxruntime as ort
import soundfile as sf
import sys

# テキスト処理のための最小限のモジュール
# 注意: 本番環境では適切なテキスト処理ライブラリを使用してください

class MMVCInference:
    def __init__(self, model_path, config_path):
        # 設定読み込み
        with open(config_path, 'r', encoding='utf-8') as f:
            self.config = json.load(f)
        
        # ONNX Runtime セッション作成
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        self.session = ort.InferenceSession(model_path, providers=providers)
        
        print(f"モデル読み込み完了: {{model_path}}")
        print(f"使用プロバイダー: {{self.session.get_providers()}}")
    
    def text_to_sequence(self, text):
        # 簡易的なテキスト→音素変換
        # 実際の使用では適切なテキスト処理を実装してください
        # ここでは例として文字をそのまま数値に変換
        return [ord(c) % 100 for c in text[:20]]  # 最大20文字
    
    def synthesize(self, text, speaker_id=0, noise_scale=0.667, 
                  noise_scale_w=0.8, length_scale=1.0):
        # テキストを音素シーケンスに変換
        phonemes = self.text_to_sequence(text)
        
        # 入力データの準備
        input_dict = {{
            "x": np.array([phonemes], dtype=np.int64),
            "x_lengths": np.array([len(phonemes)], dtype=np.int64),
            "noise_scale": np.array([noise_scale], dtype=np.float32),
            "noise_scale_w": np.array([noise_scale_w], dtype=np.float32),
            "length_scale": np.array([length_scale], dtype=np.float32)
        }}
        
        if self.config.get("n_speakers", 0) > 0:
            input_dict["sid"] = np.array([speaker_id], dtype=np.int64)
        
        # 推論実行
        outputs = self.session.run(["audio"], input_dict)
        audio = outputs[0][0, 0]  # バッチとチャンネル次元を削除
        
        return audio
    
    def save_audio(self, audio, output_path):
        sf.write(output_path, audio, self.config["sampling_rate"])

# 使用例
if __name__ == "__main__":
    # 使用方法
    if len(sys.argv) < 3:
        print("使用方法: python inference.py <テキスト> <出力ファイル>")
        sys.exit(1)
    
    text = sys.argv[1]
    output_file = sys.argv[2]
    
    # 推論器の初期化
    model_path = "mmvc_model_optimized.onnx"  # または mmvc_model.onnx
    config_path = "config.json"
    
    if not os.path.exists(model_path):
        model_path = "mmvc_model.onnx"
    
    if not os.path.exists(model_path):
        print(f"モデルファイルが見つかりません: {{model_path}}")
        sys.exit(1)
    
    inference = MMVCInference(model_path, config_path)
    
    # 音声合成
    print(f"テキスト: {{text}}")
    audio = inference.synthesize(text)
    
    # 音声保存
    inference.save_audio(audio, output_file)
    print(f"音声を保存しました: {{output_file}}")
'''
    
    # スクリプトファイルの保存
    script_path = os.path.join(deploy_dir, "inference.py")
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(deployment_script)
    
    # README.mdの作成
    readme_content = f'''
# MMVC ONNX Deployment Package

このパッケージには、MMVC音声合成システムのONNXモデルとデプロイメント用ファイルが含まれています。

## ファイル構成

- `mmvc_model.onnx`: 元のONNXモデル
- `mmvc_model_optimized.onnx`: 最適化済みONNXモデル（存在する場合）
- `config.json`: モデル設定ファイル
- `inference.py`: 推論用Pythonスクリプト
- `README.md`: このファイル

## 必要な依存関係

```bash
pip install onnxruntime soundfile numpy
```

GPU使用の場合:
```bash
pip install onnxruntime-gpu soundfile numpy
```

## 使用方法

### コマンドライン使用

```bash
python inference.py "こんにちは" output.wav
```

### Pythonスクリプトでの使用

```python
from inference import MMVCInference

# 推論器の初期化
inference = MMVCInference("mmvc_model_optimized.onnx", "config.json")

# 音声合成
audio = inference.synthesize("こんにちは")

# 音声保存
inference.save_audio(audio, "output.wav")
```

## 注意事項

1. 本パッケージに含まれる `inference.py` のテキスト処理は簡易的なものです。
2. 実際の本番環境では、適切な日本語テキスト処理ライブラリ（pyopenjtalk等）を使用してください。
3. GPUを使用する場合は、適切なCUDAバージョンがインストールされていることを確認してください。

## モデル情報

- サンプリング周波数: {config['sampling_rate']} Hz
- モデル名: {config['model_name']}
- スピーカー数: {config.get('n_speakers', 1)}
'''
    
    readme_path = os.path.join(deploy_dir, "README.md")
    with open(readme_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)
    
    print(f"\nデプロイメントパッケージを作成しました: {deploy_dir}")
    print("含まれるファイル:")
    for file in os.listdir(deploy_dir):
        print(f"  - {file}")

# デプロイメントパッケージの作成
create_deployment_package()

## まとめ

このノートブックでは以下の作業を行いました：

1. **モデルの読み込み**: 訓練済みPyTorchモデルの読み込み
2. **ONNX エクスポート**: 推論専用モデルのONNX形式への変換
3. **モデル検証**: ONNX モデルの構造と精度の確認
4. **最適化**: ONNX モデルの簡略化と最適化
5. **推論テスト**: ONNX Runtime での推論動作確認
6. **ベンチマーク**: PyTorchとONNXの性能比較
7. **デプロイメント**: 本番環境用パッケージの作成

## 次のステップ

1. 生成されたONNXモデルを本番環境でテスト
2. 必要に応じてモデルのさらなる最適化
3. 異なるプラットフォームでの動作確認
4. エッジデバイス向けの軽量化（INT8量子化等）

## トラブルシューティング

- **エクスポートエラー**: モデル構造を確認し、ONNX対応オペレータを使用してください
- **精度の問題**: 動的次元の設定や数値精度を確認してください  
- **性能の問題**: 使用するONNX Runtimeプロバイダーを確認してください
- **メモリエラー**: モデルサイズやバッチサイズを調整してください